In [ ]:
import boto3
from sagemaker.core.helper.session_helper import get_execution_role
from dotenv import load_dotenv
load_dotenv()
import os
sm_client = boto3.client(service_name='sagemaker')
runtime_sm_client = boto3.client(service_name='sagemaker-runtime')

account_id = boto3.client('sts').get_caller_identity()['Account']
region = boto3.Session().region_name


In [ ]:
from time import gmtime, strftime

model_name = 'cloth-detection'
container = '{}.dkr.ecr.{}.amazonaws.com/{}:latest'.format(account_id, region, model_name.replace("-", "_"))
instance_type = 'ml.g4dn.xlarge'

print('Model name: ' + model_name)
print('Container image: ' + container)

container = {
    'Image': container,
    'Environment': {
        'SAGEMAKER_CONTAINER_LOG_LEVEL': '10',
        'ENV': os.getenv('ENV'),
        'AWS_ACCESS_KEY_ID': os.getenv('AWS_ACCESS_KEY_ID'),
        'AWS_SECRET_ACCESS_KEY': os.getenv('AWS_SECRET_ACCESS_KEY'),
        'AWS_DEFAULT_REGION': os.getenv('AWS_DEFAULT_REGION'),
        'MLFLOW_TRACKING_URI': os.getenv('MLFLOW_TRACKING_URI')
    }
}

create_model_response = sm_client.create_model(
    ExecutionRoleArn = os.getenv('SAGEMAKER_EXECUTION_ROLE_ARN'),
    ModelName = model_name,
    # add env variables.
    Containers = [container])

print("Model Arn: " + create_model_response['ModelArn'])

In [ ]:
endpoint_config_name = 'cloth-detection-config'
print('Endpoint config name: ' + endpoint_config_name)

create_endpoint_config_response = sm_client.create_endpoint_config(
    EndpointConfigName = endpoint_config_name,
    ProductionVariants=[{
        'InstanceType': instance_type,
        'InitialInstanceCount': 1,
        'InitialVariantWeight': 1,
        'ModelName': model_name,
        'VariantName': 'AllTraffic'}])

print("Endpoint config Arn: " + create_endpoint_config_response['EndpointConfigArn'])

In [ ]:
import time

endpoint_name = 'cloth-detection-endpoint'
print('Endpoint name: ' + endpoint_name)

create_endpoint_response = sm_client.create_endpoint(
    EndpointName=endpoint_name,
    EndpointConfigName=endpoint_config_name)
print('Endpoint Arn: ' + create_endpoint_response['EndpointArn'])

resp = sm_client.describe_endpoint(EndpointName=endpoint_name)
status = resp['EndpointStatus']
print("Endpoint Status: " + status)

print('Waiting for {} endpoint to be in service...'.format(endpoint_name))
waiter = sm_client.get_waiter('endpoint_in_service')
waiter.wait(EndpointName=endpoint_name)

In [ ]:
resp = sm_client.describe_endpoint(EndpointName=endpoint_name)
print("Status:", resp["EndpointStatus"])
print("FailureReason:", resp.get("FailureReason"))
print(resp)


In [ ]:
import json, uuid

with open("000121.jpg", "rb") as f:
    file_bytes = f.read()

boundary = f"----Boundary{uuid.uuid4().hex}"
body = (
    f"--{boundary}\r\n"
    f'Content-Disposition: form-data; name="file"; filename="test_image.jpg"\r\n'
    f"Content-Type: image/jpeg\r\n\r\n"
).encode() + file_bytes + f"\r\n--{boundary}--\r\n".encode()

response = runtime_sm_client.invoke_endpoint(
    EndpointName=endpoint_name,
    ContentType=f"multipart/form-data; boundary={boundary}",
    Body=body,
)

result = json.loads(response["Body"].read().decode())
print(result)


In [ ]:
sm_client.delete_endpoint(EndpointName=endpoint_name)
sm_client.delete_endpoint_config(EndpointConfigName=endpoint_config_name)
sm_client.delete_model(ModelName=model_name)